# <center> **Data Pre-Processing** 🏭

---

### **Define helpers**

In [1]:
# define functions to use later...

def clone_folder(source_folder, destination_folder):
    """
    Clone source_folder to destination_folder.
    """
    import shutil

    # Copy the source folder to the destination
    shutil.copytree(source_folder, destination_folder)

    print(f'Cloned {source_folder} to {destination_folder}')

def unzip_file(zip_file_path, extract_to_path):
    """
    Unzip zip_file_path and extract its content to extract_to_path.
    """
    import zipfile
    import os

    # Ensure the extraction directory exists
    os.makedirs(extract_to_path, exist_ok=True)

    # Open the zip file
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # Extract all the contents into the specified directory
        zip_ref.extractall(extract_to_path)

    print(f'Extracted all files to {extract_to_path}')

def copy_n_files(source_folder, destination_folder, exclude_files=None, n=None, verbose=False):
    """
    Copy first n (if n is None, then all) files from source_folder and paste them in destination_folder.
    Files are copied with unique names to avoid repetion and overwriting.

    Here is an example:
    Assume we have this directory structure:
    /path/to/data_dir1/
    sub_dir1/
        image1.jpg
        image2.jpg
        image3.jpg

    After running the code (with exclude_files=[image3.jpg]),
    the destination directory (/path/to/destination) will look like this:
    /path/to/destination/
    sub_dir1/
        data_dir1-sub_dir1-image1.jpg
        data_dir1-sub_dir1-image2.jpg
    """
    import os
    import shutil
    import random

    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    files = os.listdir(source_folder)
    if n is not None: files = files[:n]
    if exclude_files is None: exclude_files = []

    for file_name in files:
        source_file = os.path.join(source_folder, file_name)
        if os.path.isfile(source_file):
            # Exclude files
            if file_name in exclude_files: continue

            # Add a unique prefix based on the source_folder
            unique_prefix = source_folder.replace(os.sep, '-')
            unique_file_name = f"{unique_prefix}-{file_name}"
            destination_file = os.path.join(destination_folder, unique_file_name)

            # Copy source_file and paste it in destination_folder with unique name
            shutil.copy2(source_file, destination_file)

            if verbose: print(f"Copied {source_file} to {destination_file}")

    if not verbose: 
        if exclude_files: print(f"{len(files)-len(exclude_files)} files copied from {source_folder} to {destination_folder}")
        else: print(f"{len(files)} files copied from {source_folder} to {destination_folder}")

def rename_folders_in_directory(directory):
    """
    Modify all folder names in directory by using the title method.
    """
    import os

    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)
        if os.path.isdir(folder_path):
            os.rename(folder_path, os.path.join(directory, folder_name.title()))

def create_folders_from_dict(base_path, folder_structure):
    """
    Create folders and subfolders from a dictionary structure.
    """
    import os

    for folder, subfolders in folder_structure.items():
        # Create the main folder
        folder_path = os.path.join(base_path, folder)
        os.makedirs(folder_path, exist_ok=True)

        # If there are subfolders, create them recursively
        if isinstance(subfolders, dict):
            create_folders_from_dict(folder_path, subfolders)

def delete_random_files(folder_path, n):
    import os
    import random

    try:
        # Get a list of all files in the folder
        files = [file for file in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, file))]

        # Check if the number of files to delete is greater than the number of files available
        if n > len(files):
            raise ValueError("Number of files to delete is greater than the number of files available")

        # Select n random files
        files_to_delete = random.sample(files, n)

        # Delete the selected files
        removed_files=[]
        for file in files_to_delete:
            file_path = os.path.join(folder_path, file)
            os.remove(file_path)
            removed_files.append(file)

        print(f"{n} files deleted successfully from {folder_path}")
        return removed_files

    except Exception as e:
        print(f"An error occurred: {e}")

def delete_random_files2(folder_path, n, seed=None):
    import os
    import random

    # set the seed for reproducibility, if provided
    if seed: random.seed(seed)

    # get a list of all files in the folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

    # check if we need to delete any files
    if len(files) <= n:
        print("There are already fewer or equal to n files in the folder.")
        return []

    # calculate the number of files to delete
    files_to_delete_count = len(files) - n

    # randomly select files to delete
    files_to_delete = random.sample(files, files_to_delete_count)

    # Delete the selected files
    removed_files=[]
    for file in files_to_delete:
        os.remove(os.path.join(folder_path, file))
        removed_files.append(file)

    print(f"{files_to_delete_count} files deleted successfully from {folder_path}")
    return removed_files

---
### **Data Pipeline**

In [2]:
import os

# define dirs
DIR1 = "raw-data/"    # DIR1: where raw data was downloaded from S3 
DIR2 = "train-data/"  # DIR2: where pre-processed data will go

# set some parameters for pre-processing
N = 1000           # N: number of images per class in train set
SEED = 42          # SEED: random seed for reproducibility
MIN_SIZE = 150     # MIN_SIZE: min height/width allowed
MAX_SIZE = 1000    # MIN_SIZE: max height/width allowed

# list of class names to build the train set
brands = ['Bmw', 'Mercedes-Benz', 'Hyundai', 'Kia', 'Toyota']

# list folder-names with raw images in DIR1 (data sources)
sources = [os.path.join(DIR1, name) for name in os.listdir(DIR1) if os.path.isdir(os.path.join(DIR1, name)) and name!='.ipynb_checkpoints']

# BY HAND: see that all data sources have the same class names as in the brands list

In [3]:
# create folder and subdirectories (all empty for now)
folder_dict = {
    DIR2: {sub_folder: {} for sub_folder in brands}
}

create_folders_from_dict("", folder_dict)

In [4]:
# copy raw images from each data source to DIR2
import shutil

for source in sources:
    for brand in brands:
        # get files to exclude from TXT file
        try:
            with open(os.path.join(source, f"{brand}.txt"), 'r') as file:
                content = file.read()
            exclude_files = content.split()
        except FileNotFoundError:
            exclude_files=None

        # copy n files from source to destination folder
        try:
            source_folder = os.path.join(source, brand)
            destination_folder = os.path.join(DIR2, brand)
            copy_n_files(source_folder, destination_folder, exclude_files)
        except FileNotFoundError:
            pass

911 files copied from raw-data/source-4/Bmw to train-data/Bmw
836 files copied from raw-data/source-4/Mercedes-Benz to train-data/Mercedes-Benz
1033 files copied from raw-data/source-4/Hyundai to train-data/Hyundai
644 files copied from raw-data/source-4/Kia to train-data/Kia
863 files copied from raw-data/source-4/Toyota to train-data/Toyota
456 files copied from raw-data/source-1/Bmw to train-data/Bmw
449 files copied from raw-data/source-1/Mercedes-Benz to train-data/Mercedes-Benz
498 files copied from raw-data/source-1/Hyundai to train-data/Hyundai
591 files copied from raw-data/source-1/Kia to train-data/Kia
500 files copied from raw-data/source-1/Toyota to train-data/Toyota
178 files copied from raw-data/source-3/Bmw to train-data/Bmw
169 files copied from raw-data/source-3/Mercedes-Benz to train-data/Mercedes-Benz
173 files copied from raw-data/source-3/Hyundai to train-data/Hyundai
167 files copied from raw-data/source-3/Kia to train-data/Kia
166 files copied from raw-data/sour

In [5]:
from PIL import Image
import math

records = {brand:0 for brand in brands}

# remove undesired files from DIR2
for brand in brands:
    path = os.path.join(DIR2, brand)
    img_names = os.listdir(path)

    for img_name in img_names:
        path_to_img = os.path.join(path, img_name)

        # remove non-JPG files
        if not img_name.endswith((".jpg", ".jpeg")): # include .png files?
            os.remove(path_to_img)
        else:
            img = Image.open(path_to_img)
            width, height = img.size

            # remove images that are too small
            if width < MIN_SIZE or height < MIN_SIZE:
                os.remove(path_to_img)
            # remove images that are too big
            elif width > MAX_SIZE or height > MAX_SIZE:
                os.remove(path_to_img)
            # image is good
            else: pass

            # NOTE: another option is to remove images by... math.prod(img.size)

    # keep record of the number of files after the process
    records[brand] = len(os.listdir(path))

# print records
print(DIR2)
for brand in brands:
    print(f"\t{brand}: {records[brand]}")

train-data/
	Bmw: 1302
	Mercedes-Benz: 1181
	Hyundai: 1563
	Kia: 1327
	Toyota: 1454


In [6]:
# delete random files to decrease dataset size
deleted_files = []
for brand in brands:
    path = os.path.join(DIR2, brand)
    deleted_files += delete_random_files2(path, n=N, seed=SEED)

# store deleted files in test_set.txt file
with open("test_set.txt", "w") as file:
    for item in deleted_files:
        file.write(item + "\n")

print(f"\nTotal of {len(deleted_files)} imgs in test set")

302 files deleted successfully from train-data/Bmw
181 files deleted successfully from train-data/Mercedes-Benz
563 files deleted successfully from train-data/Hyundai
327 files deleted successfully from train-data/Kia
454 files deleted successfully from train-data/Toyota

Total of 1827 imgs in test set


In [7]:
import glob

# print final DIR structure
print(DIR2)
for brand in brands:
    count = len(os.listdir(os.path.join(DIR2, brand)))
    print(f"\t{brand}: {count}")

# count number of subdirs in train-images dir
folders = glob.glob(os.path.join(DIR2, '*'))
folders = [f for f in folders if os.path.isdir(f)]
num_folders = len(folders)
print(f"\nTotal of {num_folders} classes")

# count number of files in train-images dir
files = glob.glob(os.path.join(DIR2, '**', '*'), recursive=True)
files = [f for f in files if os.path.isfile(f)]
num_files = len(files)
print(f"Total of {num_files} images")

train-data/
	Bmw: 1000
	Mercedes-Benz: 1000
	Hyundai: 1000
	Kia: 1000
	Toyota: 1000

Total of 5 classes
Total of 5000 images


---